In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini masih dalam status preview release dan dapat berubah sewaktu-waktu.

## Ikhtisar
IBM&reg; Circuit function menerima [abstract PUBs](./primitive-input-output) sebagai input, dan mengembalikan nilai ekspektasi yang telah dimitigasi sebagai output. Circuit function ini mencakup pipeline otomatis dan tersesuaikan untuk membantu para peneliti fokus pada penemuan algoritma dan aplikasi.

## Deskripsi
Setelah kamu mengirimkan PUB, sirkuit abstrak dan observable-mu akan ditranspilasi secara otomatis, dieksekusi pada hardware, dan diproses pasca-eksekusi untuk mengembalikan nilai ekspektasi yang telah dimitigasi. Untuk melakukannya, ini menggabungkan alat-alat berikut:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), termasuk pemilihan otomatis transpilasi berbasis AI dan heuristik untuk menerjemahkan sirkuit abstrakmu ke sirkuit ISA yang dioptimalkan untuk hardware
- [Suppression dan mitigasi error yang diperlukan untuk komputasi skala utilitas](./error-mitigation-and-suppression-techniques), termasuk measurement dan gate twirling, dynamical decoupling, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE), dan Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), untuk mengeksekusi ISA PUBs pada hardware dan mengembalikan nilai ekspektasi yang telah dimitigasi

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Mulai
Autentikasi menggunakan [API key](http://quantum.cloud.ibm.com/)-mu dan pilih Qiskit Function sebagai berikut. (Snippet ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokalmu.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Contoh
Untuk memulai, coba contoh dasar ini:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Periksa [status](/guides/functions#check-job-status) workload Qiskit Function-mu atau ambil [hasil](/guides/functions#retrieve-results) sebagai berikut:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Hasilnya memiliki format yang sama dengan [hasil Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Input
Lihat tabel berikut untuk semua parameter input yang diterima IBM Circuit function. Bagian [Options](#options) berikutnya menjelaskan lebih detail tentang `options` yang tersedia.

| Nama      | Tipe                       | Deskripsi                                                                                                                                                                                                                         | Wajib | Default                                                                  | Contoh                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nama backend untuk melakukan query.                                                                                                                                                                                              | Ya      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Iterable dari objek PUB-like (primitive unified bloc) abstrak, seperti tuple `(circuit, observables)` atau `(circuit, observables, parameter_values)`. Lihat [Ikhtisar PUBs](/guides/primitive-input-output#overview-of-pubs) untuk informasi lebih lanjut. Sirkuit bisa abstrak (non-ISA). | Ya      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opsi input. Lihat bagian **Options** untuk detail lebih lanjut.                                                                                                                                                                                | Tidak       | Lihat bagian **Options** untuk detail.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Nama resource cloud dari instance yang akan digunakan dalam format tersebut.                                                                                                                                                                                        | Tidak       | Satu dipilih secara acak jika akunmu punya akses ke beberapa instance. | `CRN`                   |

### Options
#### Struktur
Mirip dengan Qiskit Runtime primitives, opsi untuk IBM Circuit function bisa ditentukan sebagai nested dictionary. Opsi yang umum digunakan, seperti ``optimization_level`` dan ``mitigation_level``, berada di level pertama. Opsi lain yang lebih canggih dikelompokkan ke dalam kategori berbeda, seperti ``resilience``.

#### Default
Jika kamu tidak menentukan nilai untuk sebuah opsi, nilai default yang ditentukan oleh layanan akan digunakan.

#### Mitigation level
IBM Circuit function juga mendukung `mitigation_level`. Mitigation level menentukan seberapa banyak suppression dan mitigasi error yang diterapkan pada job. Level yang lebih tinggi menghasilkan hasil yang lebih akurat, dengan pengorbanan waktu pemrosesan yang lebih lama. Tingkat pengurangan error bergantung pada metode yang diterapkan. Mitigation level mengabstraksi pilihan detail metode mitigasi dan suppression error agar pengguna bisa mempertimbangkan trade-off biaya/akurasi yang sesuai untuk aplikasinya. Tabel berikut menunjukkan metode yang bersesuaian untuk setiap level.

> **Note:** Meski namanya mirip, `mitigation_level` menerapkan teknik yang berbeda dari yang digunakan oleh `resilience_level` Estimator.

Mirip dengan ``resilience_level`` di Qiskit Runtime Estimator, ``mitigation_level`` menentukan sekumpulan opsi tersesuaikan sebagai dasar. Opsi apapun yang kamu tentukan secara manual selain mitigation level akan diterapkan di atas kumpulan opsi dasar yang didefinisikan oleh mitigation level. Oleh karena itu, pada prinsipnya, kamu bisa mengatur mitigation level ke 1, tetapi kemudian mematikan measurement mitigation, meskipun ini tidak disarankan.

| **Mitigation Level** | **Teknik** |
|:-:|:-:|
| 1 [Default] | Dynamical decoupling + measurement twirling + TREX  |
| 2 | Level 1 + gate twirling + ZNE via gate folding |
| 3 | Level 1 + gate twirling + ZNE via PEA |

Contoh berikut mendemonstrasikan pengaturan mitigation level:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Semua opsi yang tersedia
Selain `mitigation_level`, IBM Circuit function juga menyediakan sejumlah opsi canggih yang memungkinkan kamu menyesuaikan trade-off biaya/akurasi. Tabel berikut menunjukkan semua opsi yang tersedia:

| Opsi | Sub-opsi | Sub-sub-opsi | Deskripsi | Pilihan | Default |
|-|-|-|-|-|-|
| default_precision |  |  | Presisi default yang digunakan untuk PUB apapun atau panggilan `run()`<br/>yang tidak menentukan presisi sendiri.<br/>Setiap input PUB bisa menentukan presisinya sendiri. Jika metode `run()` diberi presisi, maka nilai tersebut digunakan untuk semua PUB dalam panggilan `run()` yang tidak menentukan presisinya sendiri.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Waktu eksekusi maksimum dalam detik, yang didasarkan<br/>pada penggunaan QPU (bukan wall clock time). Penggunaan QPU adalah<br/>jumlah waktu QPU didedikasikan untuk memproses job-mu. Jika sebuah job melampaui batas waktu ini, job tersebut akan dibatalkan secara paksa. | Bilangan bulat dalam detik dalam rentang [1, 10800] |  |
| mitigation_level |  |  | Seberapa banyak suppression dan mitigasi error yang diterapkan. Lihat bagian [Mitigation level](#mitigation-level) untuk informasi lebih lanjut tentang metode yang digunakan di setiap level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Seberapa banyak optimasi yang dilakukan pada sirkuit. [Level yang lebih tinggi](/guides/set-optimization) menghasilkan sirkuit yang lebih dioptimalkan, dengan pengorbanan waktu transpilasi yang lebih lama. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Apakah akan mengaktifkan dynamical decoupling. Lihat [Teknik suppression dan mitigasi error](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) untuk penjelasan metodenya.  | True/False | True |
|  | sequence_type |  | Urutan dynamical decoupling yang digunakan.<br/>* `XX`: gunakan urutan `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: gunakan urutan `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: gunakan urutan<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Apakah akan menerapkan gate twirling Clifford 2-qubit. | True/False | False |
|  | enable_measure |  | Apakah akan mengaktifkan twirling pada measurement. | True/False | True |
| resilience | measure_mitigation |  | Apakah akan mengaktifkan metode mitigasi error measurement TREX. Lihat [Teknik suppression dan mitigasi error](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) untuk penjelasan metodenya.  | True/False | True |
|  | zne_mitigation |  | Apakah akan mengaktifkan metode mitigasi error Zero Noise Extrapolation. Lihat [Teknik suppression dan mitigasi error](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) untuk penjelasan metodenya.  | True/False | False |
|  | zne | amplifier | Teknik yang digunakan untuk memperkuat noise. Salah satu dari: <br/> - `gate_folding` (default) menggunakan gate folding 2-qubit untuk memperkuat noise. Jika faktor noise mengharuskan memperkuat hanya sebagian gate, maka gate-gate tersebut dipilih secara acak.<br/><br/> - `gate_folding_front` menggunakan gate folding 2-qubit untuk memperkuat noise. Jika faktor noise mengharuskan memperkuat hanya sebagian gate, maka gate-gate tersebut dipilih dari bagian depan DAG circuit yang terurut secara topologis.<br/><br/> - `gate_folding_back` menggunakan gate folding 2-qubit untuk memperkuat noise. Jika faktor noise mengharuskan memperkuat hanya sebagian gate, maka gate-gate tersebut dipilih dari bagian belakang DAG circuit yang terurut secara topologis.<br/><br/> - `pea` menggunakan teknik yang disebut Probabilistic error amplification (PEA) untuk memperkuat noise. Lihat [Teknik suppression dan mitigasi error](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) untuk penjelasan metodenya.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Faktor noise yang digunakan untuk amplifikasi noise. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Faktor noise untuk mengevaluasi model ekstrapolasi fit. Opsi ini tidak mempengaruhi eksekusi atau fitting model dengan cara apapun; ini hanya menentukan titik di mana objek `extrapolator` dievaluasi untuk dikembalikan dalam field data yang disebut `evs_extrapolated` dan `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Apakah akan mengaktifkan metode mitigasi error Probabilistic Error Cancellation. Lihat [Teknik suppression dan mitigasi error](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) untuk penjelasan metodenya.  | True/False | False |
|  | pec | max_overhead | Overhead sampling sirkuit maksimum yang diizinkan, atau `None` untuk tanpa batas. | None/ integer >1 | 100 |

Dalam contoh berikut, mengatur mitigation level ke 1 awalnya mematikan mitigasi ZNE, tetapi mengatur `zne_mitigation` ke `True` menimpa pengaturan relevan dari `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Output
Output dari Circuit function adalah [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), yang berisi dua field:

- Satu atau lebih objek [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Ini bisa diindeks langsung dari `PrimitiveResult`.

- Metadata tingkat job.

Setiap `PubResult` berisi field `data` dan `metadata`.

- Field `data` berisi setidaknya sebuah array nilai ekspektasi (`PubResult.data.evs`) dan sebuah array standard error (`PubResult.data.stds`). Field ini juga bisa berisi lebih banyak data, tergantung pada opsi yang digunakan.

- Field `metadata` berisi metadata tingkat PUB (`PubResult.metadata`).

Snippet kode berikut mendeskripsikan format `PrimitiveResult` (dan `PubResult` terkait).